In [61]:
fipath = "./igrf14coeffs.txt"
fopath = "../libs/igrf/src/coeffs.hpp"

In [ ]:
def triangle(n, m): # calculates triangular number from two given values
    return int(n*(n+1)/2 - 1 + m)

In [63]:
def parse_headers(line:str):
    epochs = {} # dict with key=epoch/0 for sv, value=index in a row
    for i, item in enumerate(line):
        if (item in ['g/h', 'm', 'n']): # skip first three items
            continue
        else:
            try:
                epochs[int(float(item))] = i # epochs
            except:
                epochs[0] = i # secular var
    return epochs

In [64]:
def make_coeffs(epochs:dict, nepochs=14): # populates coeffs dict
    coeffs = {}
    for epoch in epochs.keys():
        coeffs[epoch] = [[0, 0] for i in range(triangle(nepochs-1, nepochs-1) + 1)]
    return coeffs

In [65]:
def get_data(line:list, epochs:dict, coeffs:dict):
    idx_gh = 0 if line[0] == 'g' else 1
    n = int(line[1])
    m = int(line[2])
    for year in epochs.keys():
        idx_year = epochs[year]
        value = float(line[idx_year])
        coeffs[year][triangle(n, m)][idx_gh] = value  

In [66]:
def format_coeffs(coeffs:list):
    coeffs_str = ""
    for i, (n, m) in enumerate(coeffs):
        coeffs_str += " " * 12
        coeffs_str += "{{{}, {}}}".format(n, m)
        coeffs_str += ",\n" if (i != len(coeffs)-1) else "\n"
    return coeffs_str

In [67]:
epochs = dict()
coeffs = dict()
with open(fipath, 'r') as f:
    lines = f.readlines()
    are_epochs = False
    for line in lines:
        line = line.split(' ')
        line = [elem.rstrip() for elem in line if elem != '']
        if (line[0] == 'g/h'):
            epochs = parse_headers(line)
            coeffs = make_coeffs(epochs, 14)
            are_epochs = True
        if (are_epochs & (line[0] in ['g', 'h'])):
            get_data(line, epochs, coeffs)



In [78]:
with open(fopath, 'w') as f:
    f.write("#pragma once\n#include <map>\n#include <vector>\n\n")
    f.write("using Coeffs = std::map<int, std::vector<std::pair<double, double>>>;\n\n")
    f.write("namespace igrf::constants {\n")
    f.write("    const Coeffs COEFFS = {\n")
    epochs = coeffs.keys()
    for i, epoch in enumerate(epochs):
        padding_epoch = " " * 8
        epochs_str = padding_epoch + "{" + str(epoch) + ", {\n";
        epochs_str += format_coeffs(coeffs[epoch])
        epochs_str += padding_epoch
        epochs_str += "}},\n" if (i != len(epochs) - 1) else "}}\n"
        f.write(epochs_str)
    f.write("    };\n")
    f.write("}")